# **Part 1:** Import Libraries, Load & Preprocess Data

In [9]:
# Import necessary libaries
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tensorflow.keras.regularizers import l2 # Needed for L2 regularization task

import matplotlib.pyplot as plt
from google.colab import files

# Random seed for reproduceability
tf.random.set_seed(42)

# Load the datasets
files.upload()
print("Loading datasets...")
train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")

# Seperate the Features (X) and Labels (y)
X_train = train.drop("label", axis=1).values
y_train = train["label"].values

X_test = test.drop("label", axis=1).values
y_test = test["label"].values

# Normalize the pixel values (scale them from 0-255 to 0-1)
X_train = X_train / 255.0
X_test = X_test / 255.0

# Create the validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# One-Hot Encode the labels (required for categorical_crossentropy)
Y_train = to_categorical(y_train, num_classes=25)
Y_val = to_categorical(y_val, num_classes=25)
Y_test = to_categorical(y_test, num_classes=25)


print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Number of classes: {len(np.unique(y_train))}")



Saving sign_mnist_test.csv to sign_mnist_test (5).csv
Saving sign_mnist_train.csv to sign_mnist_train (5).csv
Loading datasets...
Training set shape: (21964, 784)
Validation set shape: (5491, 784)
Test set shape: (7172, 784)
Number of classes: 24


# **Part 2:** Build the MLP Model Function

In [11]:
def build_mlp_model(n_hidden_layers, n_neurons, activation, learning_rate, optimizer_type, l2_reg = None):

  model = Sequential()

  # Define the kernel regularizer based on the l2_reg parameter
  regularizer = l2(l2_reg) if l2_reg else None

  # Loop to add the specified number of Hidden Layers
  for i in range(n_hidden_layers):
        model.add(Dense(n_neurons, activation = activation, kernel_regularizer=regularizer))

  # Add the final Output Layer
  model.add(Dense(25, activation = 'softmax'))

  # Choose an optimizer
  if optimizer_type == 'Adam':
      optimizer = Adam(learning_rate=learning_rate)
  elif optimizer_type == 'SGD':
      optimizer = SGD(learning_rate=learning_rate)
  else:
      raise ValueError("Invalid optimizer type")


  # Compile the model
  model.compile(optimizer, loss = 'categorical_crossentropy', metrics = ['accuracy'])

  return model

# **Part 3:** Train Using Varying Parameter Architecture. Evaluate Accuracy